# YuNet — Face Detection

A ~230 KB face detector that returns boxes + 5 landmarks, driving the FFHQ alignment for restoration.

**Model:** `yunet.onnx`  |  **Feature:** Face detection (5 landmarks)

Runs on CPU or GPU. Cells: setup → data → load → test → benchmark → accuracy → training notes → copyright.


## 1. Setup - code + dependencies


In [ ]:
import os, sys, subprocess, time, io, urllib.request, platform
from pathlib import Path
REPO = 'https://github.com/Zubair-hussain/Xovato-cpu-ai-studio.git'
if Path('app').exists(): ROOT = Path('.').resolve()
elif Path('../app').exists(): ROOT = Path('..').resolve()
else:
    subprocess.run(['git','clone','--depth','1',REPO], check=True)
    ROOT = Path('Xovato-cpu-ai-studio').resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','pillow','numpy','onnxruntime','soundfile'], check=False)
print('root:', ROOT, '| python', platform.python_version())

## 2. Data (test set)


In [ ]:
print('YuNet needs a REAL face photo - synthetic shapes have no faces.')
FACE_IMAGE = ''  # set to an uploaded portrait path, e.g. /content/face.jpg
from pathlib import Path as _P
have = bool(FACE_IMAGE) and _P(FACE_IMAGE).exists()
print('face image set:', have)

## 3. Download & load the model


In [ ]:
def fetch(url, dest, min_bytes):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print('present', dest.name, f'{dest.stat().st_size/1e6:.1f}MB'); return dest
    print('downloading', dest.name, '...'); urllib.request.urlretrieve(url, dest)
    assert dest.stat().st_size >= min_bytes, 'file too small (LFS pointer?)'
    print('ok', f'{dest.stat().st_size/1e6:.1f}MB'); return dest

yunet = fetch('https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx',
              ROOT/'models'/'face'/'yunet.onnx', 100_000)

## 4. Inference test


In [ ]:
from app.face_restoration import detect_faces
from app import image_processing as ip
if have:
    image = ip.load_image(_P(FACE_IMAGE).read_bytes())
    faces = detect_faces(image, yunet)
    print(f'{len(faces)} face(s); scores', [round(f.score,2) for f in faces])
else:
    print('Set FACE_IMAGE above to a real portrait, then re-run.')

## 5. Benchmark (latency)


In [ ]:
if have:
    image = ip.load_image(_P(FACE_IMAGE).read_bytes()); N=10
    t=time.perf_counter()
    for _ in range(N): detect_faces(image, yunet)
    print(f'{(time.perf_counter()-t)/N*1000:.0f} ms/detection')
else: print('need a face image to benchmark')

## 6. Accuracy signal


In [ ]:
print('Reported: ~0.834 mAP (easy) on WIDER FACE (OpenCV Zoo / libfacedetection).')
print('To measure: download WIDER FACE val, run detection, compute mAP per the toolkit.')
if have:
    faces = detect_faces(ip.load_image(_P(FACE_IMAGE).read_bytes()), yunet)
    print('confidence of detections on your image:', [round(f.score,3) for f in faces])

## 7. Training / fine-tuning notes

YuNet was trained on **WIDER FACE**. It is a compact CNN; retraining needs the libfacedetection training code and a GPU. For most uses the released model is enough - this notebook evaluates it rather than retraining it.


## 8. Copyright


In [ ]:
# ====================================================================
# COPYRIGHT NOTICE - DO NOT REMOVE
# ====================================================================
# (c) Zubair Hussain - Xovato AI Studio.
# These notebooks, the benchmark/accuracy harness, and the surrounding
# tooling are the work of Zubair Hussain. You may use and modify them for
# your own projects; keep this attribution intact, including in the
# background-removal notebook and any derived outputs.
print('(c) Zubair Hussain - Xovato AI Studio')